In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as plotly
from scipy.stats import norm


In [6]:
def black_scholes_option_price(
    S0,
    K,
    T,
    r,
    sigma,
    option_type,
):
    d1=(np.log(S0/K)+T*(r+sigma**2/2))/(sigma*np.sqrt(T))
    d2=d1-sigma*np.sqrt(T)
    
    call=(S0*norm.cdf(d1))-(K*np.exp(-r*T)*norm.cdf(d2))

    if option_type=="call":
        return call
    
    elif option_type=="put":
        return call+K*np.exp(-r*T)-S0
    else:
        raise ValueError ("option_type must be 'call' or 'put'")





In [16]:
def plotly_option_vs_stock(K, T, r, sigma, option_type="call",n_min=1,n_max=1000):
    liste_S0=[]
    liste_price=[]

    for price in range(n_min,n_max+1,1):
        liste_S0.append(price)
        liste_price.append(black_scholes_option_price(price,K,T,r,sigma,option_type))
    

    line={
        "color":"blue",
        "width":3
    }

    fig=plotly.Figure()
    fig.add_scatter(name="Option Price vs Stock Price",x=liste_S0,y=liste_price,line=line)

    fig.update_layout(
        xaxis_title="Stock Price (S0)",
        yaxis_title="Option Price",
        showlegend=True,
        template="plotly_white",
        hovermode="x unified"
    )
    return fig



In [8]:
def bs_greeks_df(S0, K, T, r, sigma, option_type="call"):

    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    gamma = norm.pdf(d1) / (S0 * sigma * np.sqrt(T))
    vega = S0 * norm.pdf(d1) * np.sqrt(T) / 100
    # Vega /100 = impact pour +1 point de volatilité, ex: 20% -> 21%

    if option_type == "call":
        delta = norm.cdf(d1)

        theta_annual = (
            -S0 * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        )

        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100

    elif option_type == "put":
        delta = norm.cdf(d1) - 1

        theta_annual = (
            -S0 * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
        )

        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    else:
        raise ValueError("option_type must be 'call' or 'put'")

    theta_daily = theta_annual / 365

    return pd.DataFrame(
        {
            "Greek": ["Delta", "Gamma", "Theta", "Vega", "Rho"],
            "Value": [delta, gamma, theta_daily, vega, rho],
            "Unit": [
                "per +1 unit of S0",
                "per +1 unit of S0",
                "per calendar day",
                "per +1 volatility point",
                "per +1 interest rate point",
            ],
        }
    )

In [9]:
def greeks_S0(K, T, r, sigma, option_type="call",n_min=1,n_max=1000):
    liste_S0=[]
    liste_delta=[]
    liste_gamma=[]
    liste_theta=[]
    liste_vega=[]
    liste_rho=[]

    for price in range(n_min,n_max+1,1):
        df=bs_greeks_df(price, K, T, r, sigma, option_type=option_type)
        liste_S0.append(price)
        liste_delta.append(df.loc[df["Greek"]=="Delta","Value"].iloc[0])
        liste_gamma.append(df.loc[df["Greek"]=="Gamma","Value"].iloc[0])
        liste_theta.append(df.loc[df["Greek"]=="Theta","Value"].iloc[0])
        liste_vega.append(df.loc[df["Greek"]=="Vega","Value"].iloc[0])
        liste_rho.append(df.loc[df["Greek"]=="Rho","Value"].iloc[0])
    
    dico={
        "Stock Price":liste_S0,
        "Delta":liste_delta,
        "Gamma":liste_gamma,
        "Theta":liste_theta,
        "Vega":liste_vega,
        "Rho":liste_rho,
    }

    df=pd.DataFrame(dico)
    return df
        


In [10]:
def plotly_greeks_S0(df,K):
    
    dico_graphs={}
    df=df.copy()
    df=df.set_index("Stock Price")
    greeks=df.columns.tolist()

    line={
        "color":"blue",
        "width":1
    }

    for greek in greeks:
        serie=df[greek]
        fig=plotly.Figure()
        fig.add_scatter(name=greek,x=serie.index,y=serie.values,mode="lines+markers",line=line)
        fig.add_vline(
            x=K,
            line_color="grey",
            line_dash="dash",
            annotation_text="Strike",
            annotation_position="top"
        )
        fig.update_layout(
            xaxis_title="Stock Price (S0)",
            yaxis_title=greek,
            showlegend=True,
            template="plotly_white",
            hovermode="x unified"
        )
        dico_graphs[greek]=fig
    
    return dico_graphs
        

In [11]:
def binomial_option_price(
    S0,
    K,
    T,
    r,
    sigma,
    N,
    option_type,
    exercise_style,
):
    """
    Price une option avec un arbre binomial CRR.

    Paramètres
    ----------
    S0 : float
        Prix spot du sous-jacent.
    K : float
        Strike.
    T : float
        Maturité en années.
    r : float
        Taux sans risque annualisé en continu.
    sigma : float
        Volatilité annualisée.
    N : int
        Nombre d'étapes de l'arbre.
    option_type : str
        "call" ou "put".
    exercise_style : str
        "european" ou "american".

    Retour
    ------
    float
        Prix de l'option.
    """


    option_type = option_type.lower()
    exercise_style = exercise_style.lower()

    if N <= 0:
        raise ValueError("N must be strictly positive.")
    if T <= 0:
        raise ValueError("T must be strictly positive.")
    if sigma <= 0:
        raise ValueError("sigma must be strictly positive.")
    if option_type not in ["call", "put"]:
        raise ValueError("option_type must be 'call' or 'put'.")
    if exercise_style not in ["european", "american"]:
        raise ValueError("exercise_style must be 'european' or 'american'.")


    dt=T/N
    discount=np.exp(-r*dt)
    u=np.exp(sigma*np.sqrt(dt)) 
    d=1/u
    p=(np.exp(r*dt)-d)/(u-d)

    #final payoffs calculations

    possible_nb_downs=np.arange(N+1)
    prices=S0*d**possible_nb_downs*u**(N-possible_nb_downs)

    if option_type=="call":
        values=np.maximum(prices-K,0)
    elif option_type=="put":
        values=np.maximum(K-prices,0)
    else:
        raise ValueError("option_type must be call or put")
    
    #loop for each node level

    for step in range(N-1,-1,-1):

        if exercise_style=="european":

            values =discount*(p*values[:-1]+(1-p)*values[1:])
        
        elif exercise_style=="american":

            values =discount*(p*values[:-1]+(1-p)*values[1:])

            if option_type=="call":
                step_nb_downs=np.arange(step+1)
                step_prices=S0*d**step_nb_downs*u**(step-step_nb_downs)
                exercice_value=np.maximum(step_prices-K,0)
                values=np.maximum(exercice_value,values)


            elif option_type=="put":
                step_nb_downs=np.arange(step+1)
                step_prices=S0*d**step_nb_downs*u**(step-step_nb_downs)
                exercice_value=np.maximum(K-step_prices,0)
                values=np.maximum(exercice_value,values)
                
            else:
                raise ValueError("option_type must be call or put")
        else:
            raise ValueError("exercise_style must be american or european")
        
    return values[0]
            

In [12]:
def binomial_convergence(
    S0,
    K,
    T,
    r,
    sigma,
    option_type,
    exercise_style,
    n_min=1,
    n_max=100,
    step=1,
):
    """
    Calcule le prix binomial pour plusieurs nombres d'étapes N.
    Sert à visualiser la convergence du modèle.
    """

    rows = []

    for N in range(n_min, n_max + 1, step):
        price = binomial_option_price(
            S0=S0,
            K=K,
            T=T,
            r=r,
            sigma=sigma,
            N=N,
            option_type=option_type,
            exercise_style=exercise_style,
        )

        rows.append(
            {
                "N": N,
                "price": price,
            }
        )

    return pd.DataFrame(rows)

In [13]:
def plotly_binomial_convergence(df_conv,bs_price):

    df_bs=df_conv.copy()
    df_bs["price"]=bs_price

    fig=plotly.Figure()
    line_bin={
        "color":"blue",
        "width":2
    }
    line_bs={
        "color":"red",
        "width":3
    }
    markers={
        "color":"blue",
        "size":5
    }

    fig.add_scatter(name="Binomial Pricing",x=df_conv["N"],y=df_conv["price"],mode="lines+markers",line=line_bin,marker=markers)
    fig.add_scatter(name="Black Scholes Price",x=df_bs["N"],y=df_bs["price"],mode="lines",line=line_bs)
    fig.update_layout(
        xaxis_title="Nombre d'étapes N",
        yaxis_title="prix de l'option",
        showlegend=True,
        template="plotly_white",
        plot_bgcolor="white",
        hovermode="x unified"
    )

    return fig

In [17]:
plotly_option_vs_stock(
    1100,
    2,
    0.05,
    0.1,
    "call",
)

In [14]:
df_convergence=binomial_convergence(
    1000,
    1100,
    2,
    0.05,
    0.1,
    "call",
    "european",
)

bs_price=black_scholes_option_price(
    1000,
    1100,
    2,
    0.05,
    0.1,
    "call",
)

df2=greeks_S0(
    1100,
    2,
    0.05,
    0.1,
    "call",
)

In [53]:
df2

,Stock Price,Delta,Gamma,Theta,Vega,Rho
0,1,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00
1,2,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00
2,3,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00
3,4,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00
4,5,8.438117e-306,4.461576e-304,-1.585515e-307,2.230788e-305,8.406342e-307
...,...,...,...,...,...,...
995,996,5.301042e-01,2.824209e-03,-1.029702e-01,5.603322e+00,9.430327e+00
996,997,5.329262e-01,2.819794e-03,-1.033724e-01,5.605801e+00,9.486569e+00
997,998,5.357437e-01,2.815248e-03,-1.037724e-01,5.607997e+00,9.542779e+00
998,999,5.385566e-01,2.810573e-03,-1.041703e-01,5.609910e+00,9.598953e+00


In [75]:
graphs=plotly_greeks_S0(df2,1000)
graphs["Delta"]

In [49]:
#df2=df2.set_index("Stock Price")
greeks=df2.columns.tolist()
greeks
for greek in greeks:
    series=df2[greek]
    fig=plotly.Figure()
    line={
        "color":"blue",
        "width":1
    }

    fig.add_scatter(name=greek,x=series.index,y=series.values,mode="lines+markers",line=line)
    fig.update_layout(
        xaxis_title="Stock Price (S0)",
        yaxis_title=greek,
        showlegend=True,
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()


In [17]:
bs_greeks_df

,Greek,Value,Unit
0,Delta,0.541365,per +1 unit of S0
1,Gamma,0.002806,per +1 unit of S0
2,Theta,-0.104566,per calendar day
3,Vega,5.611541,per +1 volatility point
4,Rho,9.655089,per +1 interest rate point


In [80]:
plotly_binomial_convergence(df_convergence,bs_price).show()